# 🛣️ SHAP-Based Model Interpretability: XGBoost Road Safety Risk Explanations
This notebook implements SHAP (SHapley Additive exPlanations) to explain the Tuned XGBoost Classifier predictions. By applying cooperative game theory, SHAP decomposes the model predictions into individual feature contributions, highlighting the exact infrastructure elements driving road safety risk.

### 🔬 Key Interpretability Objectives:
1. **Global Feature Importance**: Evaluate which road conditions contribute most to classification across all segments.
2. **Directional Impact**: Understand how specific parameters (e.g. sight distance adequacy) shift risk predictions.
3. **Local Segment Interpretability**: Visualize step-by-step risk attribution for individual high-risk road sections.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Set styling for research figures
sns.set(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['font.size'] = 11
shap.initjs()

## 1. Load Data and Process Features

In [ ]:
df_raw = pd.read_csv('Final_Model_Ready_Safety_Data.csv')
df_ref = pd.read_csv('Refined_Target_Safety_Data.csv')

df_raw.columns = [c.strip() for c in df_raw.columns]
df_ref.columns = [c.strip() for c in df_ref.columns]

ordered_labels = ['Very Low', 'Low', 'Moderate', 'High', 'Very High']
existing_levels = [lvl for lvl in ordered_labels if lvl in df_ref['Refined_Risk_Level'].unique()]
risk_mapping = {label: i for i, label in enumerate(existing_levels)}

df_raw['Target'] = df_ref['Refined_Risk_Level'].map(risk_mapping)
df_raw['Crash_Index'] = df_ref['Refined_CI']

# Reconstruct candidate feature list
feature_cols = []
for col in df_raw.columns:
    if col.endswith('_ord') and col != 'Expert rating of crash likelihood in this km_ord':
        feature_cols.append(col)
    elif col.startswith('OH_'):
        feature_cols.append(col)
    elif col in ['Chainage_numeric', 'Infrastructure_Quality_Index', 'Visibility_Risk']:
        feature_cols.append(col)

X_clean = df_raw[feature_cols].copy().astype(float).fillna(0)
y_clf = df_raw['Target'].copy().fillna(0).astype(int)

print(f"Feature matrix dimensions: {X_clean.shape}")
print(f"Target classes: {existing_levels}")

## 2. Fit Final Optimized XGBoost Classifier
We train the model using the optimized hyperparameters obtained from our nested cross-validation pipeline.

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_clean)
X_scaled_df = pd.DataFrame(X_scaled, columns=X_clean.columns)

# Balance dataset with SMOTE for stable modeling
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_scaled, y_clf)
X_res_df = pd.DataFrame(X_res, columns=X_clean.columns)

# Best hyperparameters from validation
xgb_model = XGBClassifier(
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42,
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.75,
    colsample_bytree=1.0,
    min_child_weight=3,
    gamma=0.2,
    reg_alpha=0.01,
    reg_lambda=2.0
)

xgb_model.fit(X_res, y_res)
print("XGBoost Classifier successfully trained.")

## 3. Compute SHAP Values
Using the tree explainer optimized for tree-based models like XGBoost.

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_scaled)

# Output shape verification
if isinstance(shap_values, list):
    print(f"SHAP list elements count (number of classes): {len(shap_values)}")
    print(f"Shape of first class SHAP values: {shap_values[0].shape}")
else:
    print(f"SHAP array shape: {shap_values.shape}")

## 4. SHAP Visualizations for Research

### A. Global Feature Importance Bar Plot
This plot shows the average magnitude of feature contributions across all target classes. Features at the top are key safety parameters that dictate overall model output.

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values, 
    X_scaled_df, 
    plot_type="bar", 
    class_names=existing_levels,
    show=False
)
plt.title("Global Feature Importance: Mean Absolute SHAP Value", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_global_importance_bar.png', dpi=600, bbox_inches='tight')
plt.show()

### B. Directional SHAP Summary (Class 3: Very High Risk)
This summary dot plot shows how higher and lower levels of input features shift predictions specifically towards or away from the **Very High Risk** category.

*   **Red points** represent high feature values; **blue points** represent low feature values.
*   Points to the right of the center line indicate features that increase the probability of Very High Risk.

In [ ]:
# Determine the index of 'Very High' in existing levels
very_high_idx = existing_levels.index('Very High')

plt.figure(figsize=(10, 8))
if isinstance(shap_values, list):
    active_shap = shap_values[very_high_idx]
else:
    active_shap = shap_values[:, :, very_high_idx]

shap.summary_plot(
    active_shap, 
    X_scaled_df, 
    show=False
)
plt.title("Feature Impact Summary: Very High Risk Class", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary_dot_very_high.png', dpi=600, bbox_inches='tight')
plt.show()

### C. SHAP Dependence Plot: Sight Distance Adequacy
Illustrates how different values of the top feature "Is sight distance adequate for safe driving?" relate to the SHAP values, showing any threshold or interaction effects.

In [ ]:
top_feature = 'Is sight distance adequate for safe driving?_ord'

plt.figure(figsize=(8, 6))
shap.dependence_plot(
    top_feature, 
    active_shap, 
    X_clean,  # Using original (unscaled) values for clear x-axis interpretation
    interaction_index='auto',
    show=False
)
plt.title(f"SHAP Dependence: {top_feature} (Very High Risk)", fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_dependence_sight_distance.png', dpi=600, bbox_inches='tight')
plt.show()

## 5. Summary Explanation for Research
1. **Sight Distance Adequacy**: Shows high global importance. Lower values (inadequate sight distance) significantly shift the prediction towards *Very High Risk*.
2. **Conflict & Commercial Activity Density**: High commercial density and pedestrian conflict frequencies represent key active frictions that boost risk scores.
3. **Infrastructure Quality Index**: Acts as a stabilizing compound variable; lower physical infrastructure ratings correlate with increased accident susceptibility.